# Electromagnetic Formalization of the Pipe Geometry
**Cylindrical waveguide modes → dispersive phase retrieval connection**

This notebook derives the EM boundary conditions for a cylindrical pipe/fiber,
shows why the TD-GS pipe extension (`retrieve_phase_pipe`) is physically motivated,
and computes Poynting power, modal dispersion, and field profiles.

| § | Topic |
|---|-------|
| 1 | Maxwell's equations in cylindrical coordinates |
| 2 | TE/TM modes — boundary conditions at $r=a$ |
| 3 | LP (linearly polarized) fiber modes |
| 4 | Poynting vector — average power in each mode |
| 5 | Group velocity & GVD dispersion in the pipe |
| 6 | Phase continuity justification for `retrieve_phase_pipe` |
| 7 | Numerical mode solver (FD) |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv, kv, jvp, kvp   # Bessel functions
from scipy.optimize import brentq
from scipy.constants import c, epsilon_0, mu_0
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

# Fiber/pipe parameters (step-index, SMF-28 approximate)
a    = 4.1e-6      # core radius (m) — SMF-28 ~4.1 µm
n1   = 1.4682      # core refractive index
n2   = 1.4629      # cladding index
lam0 = 1550e-9     # free-space wavelength (m)
k0   = 2*np.pi/lam0
NA   = np.sqrt(n1**2 - n2**2)
V    = k0 * a * NA  # V-number (normalized frequency)

print(f'Core radius:      a  = {a*1e6:.2f} µm')
print(f'Core index:       n1 = {n1:.4f}')
print(f'Cladding index:   n2 = {n2:.4f}')
print(f'NA = {NA:.4f}')
print(f'V  = {V:.4f}  ({"single-mode" if V < 2.405 else "multi-mode"})')

---
## §1 · Maxwell's equations in cylindrical coordinates

For a fiber/pipe aligned along $\hat{z}$, modes have the form:
$$\mathbf{E}(r,\phi,z,t) = \mathbf{e}(r,\phi)\,e^{i(\beta z - \omega t)}$$

Maxwell's equations reduce to two scalar Helmholtz equations for $E_z$ and $H_z$:
$$\nabla_T^2 E_z + (k^2 - \beta^2)E_z = 0$$
$$\nabla_T^2 H_z + (k^2 - \beta^2)H_z = 0$$

where $\nabla_T^2 = \partial_r^2 + r^{-1}\partial_r + r^{-2}\partial_\phi^2$ is the transverse Laplacian.

**Solutions:**
- Core ($r < a$):  $J_\nu(\kappa r)e^{i\nu\phi}$
- Cladding ($r > a$): $K_\nu(\gamma r)e^{i\nu\phi}$

where $\kappa^2 = n_1^2 k_0^2 - \beta^2$ and $\gamma^2 = \beta^2 - n_2^2 k_0^2$.

In [ ]:
# Characteristic equation for LP(ν,m) modes
# LP modes: ν = azimuthal order, m = radial order
# HE/EH hybrid modes approximated as LP modes (valid for weakly guiding, Δ << 1)

Delta = (n1**2 - n2**2) / (2*n1**2)
print(f'Δ = {Delta:.5f}  (weakly guiding: Δ << 1  ✓ since {Delta:.3f} << 1)')

def char_eq_LP(U, V, nu):
    """
    LP mode characteristic equation:
        J_{ν-1}(U) / J_{ν+1}(U)  = -K_{ν-1}(W) / K_{ν+1}(W)
    or equivalently using Bessel recurrence:
        U * J_{ν+1}(U) * K_ν(W)  = W * K_{ν+1}(W) * J_ν(U)
    where W = sqrt(V^2 - U^2).
    """
    if U <= 0 or U >= V:
        return np.nan
    W = np.sqrt(max(V**2 - U**2, 1e-30))
    lhs = U * jv(nu+1, U) * kv(nu,   W)
    rhs = W * kv(nu+1, W) * jv(nu,   U)
    return lhs - rhs

# Find all LP modes for current V
def find_lp_modes(V_val):
    modes = []
    for nu in range(0, 5):
        U_vals = np.linspace(0.01, V_val - 0.01, 2000)
        f_vals = np.array([char_eq_LP(u, V_val, nu) for u in U_vals])
        # Find sign changes → roots
        for i in range(len(f_vals)-1):
            if np.isnan(f_vals[i]) or np.isnan(f_vals[i+1]):
                continue
            if f_vals[i] * f_vals[i+1] < 0:
                try:
                    U_root = brentq(char_eq_LP, U_vals[i], U_vals[i+1],
                                    args=(V_val, nu), xtol=1e-8)
                    W_root = np.sqrt(V_val**2 - U_root**2)
                    # Effective index
                    beta_norm = np.sqrt((U_root/V_val)**2 * (n1**2-n2**2) + n2**2) \
                                if V_val > 0 else n2
                    modes.append({'nu': nu, 'U': U_root, 'W': W_root,
                                  'beta_norm': beta_norm})
                except Exception:
                    pass
    return modes

modes = find_lp_modes(V)
print(f'\nLP modes found for V={V:.3f}:')
for i, m in enumerate(modes[:8]):
    print(f'  LP({m["nu"]},{i+1})?  U={m["U"]:.4f}  W={m["W"]:.4f}  β/k0≈{m["beta_norm"]:.5f}')

## §2 · TE/TM/Hybrid mode cutoff V-numbers

| Mode | Cutoff $V_c$ | Description |
|------|------------|-------------|
| LP$_{01}$ (HE$_{11}$) | 0 | Fundamental, always guided |
| LP$_{11}$ (TE$_{01}$, TM$_{01}$, HE$_{21}$) | 2.405 | First cutoff |
| LP$_{21}$ (HE$_{31}$, EH$_{11}$) | 3.832 | |
| LP$_{02}$ (HE$_{12}$) | 3.832 | |
| LP$_{31}$ (HE$_{41}$, EH$_{21}$) | 5.136 | |

In [ ]:
# Number of guided modes vs. V-number
V_arr = np.linspace(0.5, 10, 200)
n_modes_arr = [len(find_lp_modes(v)) for v in V_arr]

# Approximate formula: N ≈ V²/2 (large V limit)
N_approx = V_arr**2 / 2

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].step(V_arr, n_modes_arr, where='post', lw=2, color='steelblue', label='Exact (LP)')
axes[0].plot(V_arr, N_approx, '--', color='gray', label='$N \\approx V^2/2$')
for vc, lbl in [(2.405,'LP₁₁'),(3.832,'LP₀₂/LP₂₁'),(5.136,'LP₃₁')]:
    axes[0].axvline(vc, ls=':', color='crimson', alpha=0.7)
    axes[0].text(vc+0.05, max(n_modes_arr)*0.7, lbl, fontsize=8, color='crimson')
axes[0].axvline(V, ls='--', color='orange', lw=2, label=f'This fiber V={V:.2f}')
axes[0].set_xlabel('V-number'); axes[0].set_ylabel('Number of guided LP modes')
axes[0].set_title('Mode count vs. normalized frequency')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

# Effective index vs. V for first few modes
V_sweep = np.linspace(1.0, 8.0, 80)
for nu_plot, col, lbl in [(0,'#377eb8','LP₀₁'),(1,'#e41a1c','LP₁₁'),(2,'#4daf4a','LP₂₁')]:
    b_eff = []
    for v in V_sweep:
        ms = [m for m in find_lp_modes(v) if m['nu']==nu_plot]
        if ms:
            b_eff.append(ms[0]['beta_norm'])
        else:
            b_eff.append(np.nan)
    axes[1].plot(V_sweep, b_eff, lw=2, color=col, label=lbl)

axes[1].axhline(n1, ls='--', color='gray', alpha=0.6, label=f'$n_1$={n1}')
axes[1].axhline(n2, ls='--', color='gray', alpha=0.4, label=f'$n_2$={n2}')
axes[1].set_xlabel('V-number'); axes[1].set_ylabel('Effective index $n_{eff}$')
axes[1].set_title('Modal effective index vs. V')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

plt.suptitle('Step-index fiber modal analysis', fontweight='bold')
plt.tight_layout(); plt.show()

## §3 · LP mode field profiles

$$E_x(r, \phi) = \begin{cases}
A\, J_\nu(\kappa r)\cos(\nu\phi) & r \leq a \\
A\, \dfrac{J_\nu(\kappa a)}{K_\nu(\gamma a)} K_\nu(\gamma r)\cos(\nu\phi) & r > a
\end{cases}$$

In [ ]:
def field_profile_2d(mode, V_val, a_m, lam, n1, n2, N_grid=200):
    """2D transverse field intensity |E|² for an LP(nu, m) mode."""
    nu = mode['nu']
    U  = mode['U']
    W  = mode['W']
    kappa = U / a_m
    gamma = W / a_m
    norm_factor = jv(nu, U) / kv(nu, W) if kv(nu, W) != 0 else 1.0

    r_span = 3 * a_m
    x = np.linspace(-r_span, r_span, N_grid)
    y = np.linspace(-r_span, r_span, N_grid)
    X, Y = np.meshgrid(x, y)
    R    = np.sqrt(X**2 + Y**2)
    Phi  = np.arctan2(Y, X)

    E = np.where(
        R <= a_m,
        jv(nu, kappa * R) * np.cos(nu * Phi),
        norm_factor * kv(nu, gamma * R) * np.cos(nu * Phi)
    )
    return X, Y, np.abs(E)**2

if modes:
    n_show = min(4, len(modes))
    fig, axes = plt.subplots(1, n_show, figsize=(4*n_show, 4))
    if n_show == 1:
        axes = [axes]

    for ax, m in zip(axes, modes[:n_show]):
        X, Y, I = field_profile_2d(m, V, a, lam0, n1, n2)
        ax.imshow(I, extent=[X.min()*1e6, X.max()*1e6, Y.min()*1e6, Y.max()*1e6],
                  cmap='inferno', origin='lower', aspect='equal')
        circ = plt.Circle((0, 0), a*1e6, fill=False, color='white', lw=1.5, ls='--')
        ax.add_patch(circ)
        ax.set_title(f'LP$_{{{m["nu"]}}}$ mode\nU={m["U"]:.3f}')
        ax.set_xlabel('x (µm)'); ax.set_ylabel('y (µm)')

    plt.suptitle('LP mode intensity profiles  |E(x,y)|²', fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('No modes found for current V. Try increasing V (larger core or shorter λ).')

## §4 · Poynting vector — average power in the pipe

The time-averaged Poynting vector:
$$\langle \mathbf{S} \rangle = \frac{1}{2}\text{Re}(\mathbf{E} \times \mathbf{H}^*)$$

Axial power carried by a guided mode:
$$P = \int_0^\infty \langle S_z \rangle\, r\,dr\,d\phi = \frac{\beta}{2\omega\mu_0}\int|E_T|^2\,dA$$

**Confinement factor:** fraction of power in the core:
$$\Gamma = \frac{\int_0^a |E|^2 r\,dr}{\int_0^\infty |E|^2 r\,dr}$$

In [ ]:
def confinement_factor(mode, a_m):
    """Numerical integration of Γ = P_core / P_total for an LP mode."""
    nu = mode['nu']
    U  = mode['U']
    W  = mode['W']
    kappa = U / a_m
    gamma = W / a_m
    norm  = jv(nu, U) / kv(nu, W)

    # Radial integration — core
    r_core = np.linspace(0, a_m, 2000)
    I_core = jv(nu, kappa*r_core)**2 * r_core
    P_core = np.trapz(I_core, r_core)

    # Cladding (integrate to 10a)
    r_clad = np.linspace(a_m, 10*a_m, 5000)
    I_clad = (norm * kv(nu, gamma*r_clad))**2 * r_clad
    P_clad = np.trapz(I_clad, r_clad)

    return P_core / (P_core + P_clad)

# Confinement vs. V for LP01
V_cf = np.linspace(0.5, 5.0, 60)
Gamma_vals = []
for v in V_cf:
    ms = find_lp_modes(v)
    if ms:
        Gamma_vals.append(confinement_factor(ms[0], a))
    else:
        Gamma_vals.append(np.nan)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(V_cf, Gamma_vals, lw=2.5, color='steelblue', label='LP$_{01}$ (fundamental)')
ax.axvline(V, ls='--', color='orange', lw=2, label=f'This fiber V={V:.2f}')
ax.axhline(0.80, ls=':', color='gray', alpha=0.6, label='Γ=0.80')
ax.set_xlabel('V-number'); ax.set_ylabel('Confinement factor Γ')
ax.set_title('Power confinement in fiber core — LP₀₁')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

if modes:
    Gamma_fund = confinement_factor(modes[0], a)
    print(f'LP₀₁ confinement at V={V:.2f}: Γ = {Gamma_fund:.4f}')
    print(f'  → {Gamma_fund*100:.1f}% of optical power is in the core')

## §5 · Group velocity & GVD dispersion in the pipe

Group velocity: $v_g = d\omega/d\beta$

Group velocity dispersion (GVD):
$$\beta_2 = \frac{d^2\beta}{d\omega^2} = -\frac{\lambda^2}{2\pi c}\frac{d^2 n_{eff}}{d\lambda^2}$$

Two contributions:
1. **Material dispersion** $D_m$ — from $dn/d\lambda$ of silica (Sellmeier)
2. **Waveguide dispersion** $D_w$ — from geometry (mode confinement vs. $\lambda$)

Total: $D = D_m + D_w$ (ps/nm/km)

In [ ]:
# Sellmeier equation for fused silica (Malitson 1965)
def n_silica(lam_um):
    """Refractive index of fused silica. lam_um in micrometers."""
    l2 = lam_um**2
    return np.sqrt(1
        + 0.6961663*l2/(l2 - 0.0684043**2)
        + 0.4079426*l2/(l2 - 0.1162414**2)
        + 0.8974794*l2/(l2 - 9.896161**2)
    )

lam_um = np.linspace(0.8, 2.0, 500)
n_sil  = n_silica(lam_um)

# Material dispersion D_m = -(lambda/c) d²n/dlambda²  [ps/nm/km]
dlam   = lam_um[1] - lam_um[0]
d2n    = np.gradient(np.gradient(n_sil, dlam), dlam)
D_mat  = -(lam_um / (c*1e-3)) * d2n * 1e-3  # ps/nm/km (rough units)

# Waveguide dispersion (approximate analytic)
# D_w ≈ -(n1-n2)/(c*lam) * V * d²(Vb)/dV²  where b = normalized propagation constant
# Using Petermann approximation for LP01
def waveguide_dispersion(lam_um_arr, a_m, n1, n2):
    Dw = []
    for lu in lam_um_arr:
        lm  = lu * 1e-6
        k_0 = 2*np.pi/lm
        NA_ = np.sqrt(n1**2 - n2**2)
        V_  = k_0 * a_m * NA_
        # b(V) ≈ (1 - 1.1428/V)² for LP01, V>1  (approximate)
        b   = max(0, (1 - 1.1428/V_)**2) if V_ > 1.1 else 0.0
        db  = 2*(1 - 1.1428/V_) * (1.1428/V_**2)
        d2b = 2*(1.1428/V_**2)**2 + 2*(1-1.1428/V_)*(-2*1.1428/V_**3)
        # D_w = -(n1^2-n2^2)/(2*n1*c*lam) * V * d²(Vb)/dV²
        Vd2Vb = 2*db + V_*d2b
        Dw.append(-(n1-n2) / (c*1e-3*lu*1e-9) * V_ * Vd2Vb * 1e3)
    return np.array(Dw)

D_wg  = waveguide_dispersion(lam_um, a, n1, n2)
D_tot = D_mat + D_wg

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(lam_um*1e3, n_sil, lw=2, color='steelblue')
axes[0].axvline(1550, ls='--', color='gray', label='1550 nm')
axes[0].set_xlabel('Wavelength (nm)'); axes[0].set_ylabel('Refractive index')
axes[0].set_title('Sellmeier — fused silica')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(lam_um*1e3, D_mat, lw=2, color='steelblue',  label='Material $D_m$')
axes[1].plot(lam_um*1e3, D_wg,  lw=2, color='crimson',    label='Waveguide $D_w$')
axes[1].plot(lam_um*1e3, D_tot, lw=2.5, color='k',        label='Total $D$', ls='--')
axes[1].axhline(0,    ls='-',  color='gray', lw=0.8)
axes[1].axvline(1550, ls='--', color='gray', alpha=0.6)
axes[1].set_xlabel('Wavelength (nm)'); axes[1].set_ylabel('Dispersion D (ps/nm/km)')
axes[1].set_title('Chromatic dispersion — SMF-28 approximate')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout(); plt.show()

idx_1550 = np.argmin(np.abs(lam_um - 1.55))
print(f'At 1550 nm:  D_mat={D_mat[idx_1550]:.2f}  D_wg={D_wg[idx_1550]:.2f}  D_tot={D_tot[idx_1550]:.2f} ps/nm/km')
print(f'SMF-28 spec: D ≈ +17 ps/nm/km at 1550 nm')

## §6 · Phase continuity — justification for `retrieve_phase_pipe`

**Why is angular phase continuity valid?**

In a circular symmetric waveguide, adjacent LP modes differ only in azimuthal index $\nu$.  For a signal occupying a superposition of nearby modes, the phase $\phi(\theta, z, t)$ varies smoothly around the cylinder — the same smoothness GS exploits axially.  The wrap-around seam correction removes the accumulated global-phase drift:

$$\phi(2\pi, z, t) \approx \phi(0, z, t) + \underbrace{\delta_{\text{seam}}}_{\text{distributed equally}}$$

**When does this break down?**
- Strong modal beating with period $< \lambda_{beat}$ — phase is non-smooth between adjacent $\theta$ nodes
- Multi-mode fiber with large $\Delta n_{eff}$ between modes

In [ ]:
# Beat length between LP01 and LP11
if len(modes) >= 2:
    beta1 = modes[0]['beta_norm'] * k0
    beta2 = modes[1]['beta_norm'] * k0
    L_beat = 2*np.pi / abs(beta1 - beta2) if abs(beta1 - beta2) > 1e-10 else np.inf
    print(f'Beat length LP01-LP11: L_beat = {L_beat*100:.2f} cm')
    print(f'Phase continuity valid when signal length << {L_beat*100:.1f} cm')
else:
    print('Only one mode guided (SMF) — no intermodal beating, continuity always valid.')

# Demonstrate phase smoothness on cylinder for a bandlimited signal
N_theta = 32
theta   = np.linspace(0, 2*np.pi, N_theta, endpoint=False)

# Smooth phase on cylinder: sum of low azimuthal harmonics
rng = np.random.default_rng(42)
phi_cyl = sum(
    rng.uniform(-1,1) * np.sin(k*theta + rng.uniform(0, 2*np.pi))
    for k in range(1, 4)   # only ν=1,2,3 modes
)
phi_cyl /= np.max(np.abs(phi_cyl)) * 1.2

fig, ax = plt.subplots(subplot_kw={'projection': 'polar'}, figsize=(6, 6))
ax.plot(theta, phi_cyl + 2, lw=2, color='steelblue', label='φ(θ)')
ax.fill_between(theta, 0, phi_cyl + 2, alpha=0.2, color='steelblue')
ax.set_title('Smooth azimuthal phase on cylinder\n(low-ν mode superposition)', pad=20)
plt.tight_layout(); plt.show()
print('Phase is smooth → angular continuity correction is valid.')

## §7 · Finite-difference mode solver

Full-vector 2D FD mode solver for arbitrary index profiles.  
Discretizes $\nabla_T^2 E_z + (n^2 k_0^2 - \beta^2) E_z = 0$ on a 2D grid and finds eigenvalues $\beta$ via `scipy.sparse.linalg.eigs`.

In [ ]:
from scipy.sparse import diags, kron, eye as speye
from scipy.sparse.linalg import eigs

def fd_mode_solver(n_profile, dx, k0, n_eff_guess, n_modes=4):
    """
    2D scalar FD mode solver.

    Parameters
    ----------
    n_profile   : (Nx, Ny) array — refractive index profile
    dx          : float — grid spacing (m), assumed isotropic
    k0          : float — free-space wavenumber
    n_eff_guess : float — starting guess for effective index
    n_modes     : int — number of modes to find

    Returns
    -------
    betas   : (n_modes,) float array — propagation constants
    E_modes : (n_modes, Nx, Ny) complex array — modal field profiles
    """
    Nx, Ny = n_profile.shape
    N = Nx * Ny

    # 1D second-derivative FD matrix
    d2 = diags([1, -2, 1], [-1, 0, 1], shape=(Nx, Nx)) / dx**2
    Ix = speye(Nx); Iy = speye(Ny)

    # 2D Laplacian via Kronecker product
    Lap = kron(d2, Iy) + kron(Ix, d2)

    # Diagonal refractive index matrix: k0² n²(x,y)
    n2_flat = n_profile.ravel()**2
    D_n2    = diags(k0**2 * n2_flat)

    # Eigenvalue problem: (Lap + D_n2) E = beta² E
    A = Lap + D_n2
    sigma = (n_eff_guess * k0)**2

    try:
        vals, vecs = eigs(A.astype(complex), k=n_modes, sigma=sigma,
                         which='LM', tol=1e-8)
        betas   = np.sqrt(np.real(vals))
        E_modes = [vecs[:, i].reshape(Nx, Ny) for i in range(n_modes)]
        # Sort by descending beta
        idx     = np.argsort(-betas)
        return betas[idx], [E_modes[i] for i in idx]
    except Exception as ex:
        print(f'Solver failed: {ex}')
        return [], []


# Build step-index circular fiber profile on a grid
Ng  = 60         # grid points per side
L   = 6 * a      # window size
dx  = L / Ng
x1d = np.linspace(-L/2, L/2, Ng)
X2, Y2 = np.meshgrid(x1d, x1d)
R2 = np.sqrt(X2**2 + Y2**2)
n_prof = np.where(R2 <= a, n1, n2)

print('Running FD mode solver (may take a few seconds)...')
betas_fd, E_modes_fd = fd_mode_solver(n_prof, dx, k0, n_eff_guess=(n1+n2)/2, n_modes=6)

if len(betas_fd) > 0:
    print(f'Found {len(betas_fd)} modes:')
    for i, b in enumerate(betas_fd):
        neff = b / k0
        if n2 < neff < n1:
            print(f'  Mode {i}: β = {b:.3f}  neff = {neff:.5f}  [guided]')

    fig, axes = plt.subplots(1, min(4, len(betas_fd)), figsize=(14, 4))
    if len(betas_fd) == 1: axes = [axes]
    for ax, E, b in zip(axes, E_modes_fd, betas_fd):
        neff = b/k0
        ax.imshow(np.abs(E)**2, cmap='inferno', aspect='equal',
                  extent=[-L/2*1e6, L/2*1e6, -L/2*1e6, L/2*1e6])
        circ = plt.Circle((0,0), a*1e6, fill=False, color='white', lw=1.5, ls='--')
        ax.add_patch(circ)
        ax.set_title(f'FD mode\nneff={neff:.4f}')
        ax.set_xlabel('x (µm)')
    plt.suptitle('FD solver: |E(x,y)|² profiles', fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('No guided modes found. Adjust grid or index contrast.')

---
## Summary: EM pipe → TD-GS connection

| EM quantity | TD-GS equivalent |
|------------|------------------|
| Propagation constant $\beta(\omega)$ | Dispersion parameter $D$ |
| GVD $\beta_2 = d^2\beta/d\omega^2$ | $D_{\text{phys}}$ (ps/nm/km) |
| Modal field $E(r,\phi,z)$ | Complex envelope $E(t)$ at detector |
| Photodetector: $I = |E|^2$ | Measured intensity $I_1(t)$, $I_2(t)$ |
| Azimuthal mode index $\nu$ | Angular stack axis $\theta$ in `retrieve_phase_pipe` |
| Phase continuity ($J_\nu$ smooth) | `angular_continuity=True` in pipe retrieval |
| Average power $\langle S_z \rangle$ | Signal energy normalization for GS |

**Physical dispersion values (paper):**
- DCF arm 1: $D_1 = -695$ ps/nm → $D_{\text{norm}} = -5000$
- DCF arm 2: $D_2 = -800$ ps/nm → $D_{\text{norm}} = -5750$
- Scale: $7.19$ normalized units per ps/nm